In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn

PROJECT_ROOT = Path.cwd().parent

DATASET_DIR = (
    PROJECT_ROOT
    / "dataset"
    / "CIC-IDS2017"
    / "MachineLearningCVE"
)

print("Project root:", PROJECT_ROOT)
print("Dataset exists:", DATASET_DIR.exists())
print("CSV files:", len(list(DATASET_DIR.glob("*.csv"))))
print("Pandas version:", pd.__version__)
print("Scikit-learn version:", sklearn.__version__)

Project root: C:\atharv\college\studies\coding\project a new start\NIDS
Dataset exists: True
CSV files: 8
Pandas version: 3.1.0
Scikit-learn version: 1.9.1


In [ ]:
# Find all dataset files
csv_files = sorted(DATASET_DIR.glob("*.csv"))

# Inspect the first file without loading the entire dataset
df_sample = pd.read_csv(csv_files[0], nrows=5, low_memory=False)

# Clean whitespace around column names
df_sample.columns = df_sample.columns.str.strip()

print("File:", csv_files[0].name)
print("Sample shape:", df_sample.shape)

print("\nColumn names:")
print(df_sample.columns.tolist())

print("\nSample records:")
display(df_sample.head())

print("\nData types:")
display(df_sample.dtypes.value_counts())

File: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Sample shape: (5, 79)

Column names:
['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Cou

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,54865,3,2,0,12,0,6,6,6,0,...,20,0,0,0,0,0,0,0,0,BENIGN
1,55054,109,1,1,6,6,6,6,6,0,...,20,0,0,0,0,0,0,0,0,BENIGN
2,55055,52,1,1,6,6,6,6,6,0,...,20,0,0,0,0,0,0,0,0,BENIGN
3,46236,34,1,1,6,6,6,6,6,0,...,20,0,0,0,0,0,0,0,0,BENIGN
4,54863,3,2,0,12,0,6,6,6,0,...,20,0,0,0,0,0,0,0,0,BENIGN



Data types:


int64      74
float64     4
str         1
Name: count, dtype: int64

In [ ]:
from collections import Counter

label_counts = Counter()

for file in csv_files:
    print("Reading:", file.name)

    # Read the original header without modifying its names
    header = pd.read_csv(file, nrows=0)

    # Find the original column name that corresponds to "Label"
    label_column = next(
        (
            col for col in header.columns
            if str(col).strip().lower() == "label"
        ),
        None,
    )

    if label_column is None:
        raise ValueError(
            f"Label column not found in {file.name}"
        )

    # Read only the label column in chunks
    for chunk in pd.read_csv(
        file,
        usecols=[label_column],
        chunksize=100_000,
        low_memory=False,
    ):
        labels = chunk[label_column].astype("string").str.strip()
        label_counts.update(labels.dropna().tolist())

label_summary = pd.DataFrame(
    label_counts.items(),
    columns=["Original Label", "Count"],
).sort_values("Count", ascending=False)

display(label_summary)

print("\nTotal rows:", label_summary["Count"].sum())
print("Unique original labels:", len(label_summary))

Reading: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Reading: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Reading: Friday-WorkingHours-Morning.pcap_ISCX.csv
Reading: Monday-WorkingHours.pcap_ISCX.csv
Reading: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Reading: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Reading: Tuesday-WorkingHours.pcap_ISCX.csv
Reading: Wednesday-workingHours.pcap_ISCX.csv


,Original Label,Count
0,BENIGN,2273097
12,DoS Hulk,231073
2,PortScan,158930
1,DDoS,128027
13,DoS GoldenEye,10293
8,FTP-Patator,7938
9,SSH-Patator,5897
10,DoS slowloris,5796
11,DoS Slowhttptest,5499
3,Bot,1966



Total rows: 2830743
Unique original labels: 15


In [ ]:
def map_label(label):
    label = str(label).strip()

    if label == "BENIGN":
        return "Normal"

    if label.startswith("DoS "):
        return "DoS"

    if label == "DDoS":
        return "DDoS"

    if label == "PortScan":
        return "PortScan"

    if label in {"FTP-Patator", "SSH-Patator"}:
        return "BruteForce"

    if label.startswith("Web Attack"):
        return "WebAttack"

    if label == "Bot":
        return "Bot"

    if label == "Infiltration":
        return "Infiltration"

    if label == "Heartbleed":
        return "Heartbleed"

    return "Unknown"


label_summary["Mapped Label"] = (
    label_summary["Original Label"].map(map_label)
)

display(
    label_summary[
        ["Original Label", "Mapped Label", "Count"]
    ]
)

print("\nMapped category totals:")
display(
    label_summary.groupby("Mapped Label")["Count"]
    .sum()
    .sort_values(ascending=False)
    .rename("Count")
    .to_frame()
)

print("\nUnknown labels:",
      int((label_summary["Mapped Label"] == "Unknown").sum()))

,Original Label,Mapped Label,Count
0,BENIGN,Normal,2273097
12,DoS Hulk,DoS,231073
2,PortScan,PortScan,158930
1,DDoS,DDoS,128027
13,DoS GoldenEye,DoS,10293
8,FTP-Patator,BruteForce,7938
9,SSH-Patator,BruteForce,5897
10,DoS slowloris,DoS,5796
11,DoS Slowhttptest,DoS,5499
3,Bot,Bot,1966



Mapped category totals:


,Count
Mapped Label,
Normal,2273097
DoS,252661
PortScan,158930
DDoS,128027
BruteForce,13835
WebAttack,2180
Bot,1966
Infiltration,36
Heartbleed,11



Unknown labels: 0


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

CHUNK_SIZE = 100_000

def load_and_clean_csv(file_path):
    """Load one CSV in chunks and apply consistent basic cleaning."""
    chunks = []

    for chunk in pd.read_csv(
        file_path,
        chunksize=CHUNK_SIZE,
        low_memory=False,
    ):
        # Normalize column names
        chunk.columns = chunk.columns.str.strip()

        # Find the label column safely
        label_column = next(
            (
                col for col in chunk.columns
                if col.lower() == "label"
            ),
            None,
        )

        if label_column is None:
            raise ValueError(f"Label column missing in {file_path.name}")

        # Normalize labels and map to target categories
        chunk["Target"] = chunk[label_column].map(map_label)

        # Remove the original label column
        chunk = chunk.drop(columns=[label_column])

        # Convert feature columns to numeric
        feature_columns = [
            col for col in chunk.columns if col != "Target"
        ]

        chunk[feature_columns] = chunk[feature_columns].apply(
            pd.to_numeric, errors="coerce"
        )

        # Replace infinite values and remove rows with invalid features
        chunk[feature_columns] = chunk[feature_columns].replace(
            [np.inf, -np.inf], np.nan
        )

        chunk = chunk.dropna(
            subset=feature_columns + ["Target"]
        )

        chunks.append(chunk)

    return pd.concat(chunks, ignore_index=True)


print("Dataset loader defined successfully.")

Dataset loader defined successfully.


In [ ]:
test_df = load_and_clean_csv(csv_files[0])

print("File:", csv_files[0].name)
print("Cleaned shape:", test_df.shape)

print("\nTarget distribution:")
display(test_df["Target"].value_counts().to_frame("Count"))

print("\nMissing values:", int(test_df.isna().sum().sum()))
print("Infinite values:", int(np.isinf(
    test_df.select_dtypes(include=np.number).to_numpy()
).sum()))

print("\nFirst five rows:")
display(test_df.head())

File: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Cleaned shape: (225711, 79)

Target distribution:


,Count
Target,
DDoS,128025
Normal,97686



Missing values: 0
Infinite values: 0

First five rows:


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Target
0,54865,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,Normal
1,55054,109,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,Normal
2,55055,52,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,Normal
3,46236,34,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,Normal
4,54863,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,Normal


In [ ]:
print("Test dataset memory usage:")
print(f"{test_df.memory_usage(deep=True).sum() / (1024**2):.2f} MB")

import psutil

memory = psutil.virtual_memory()

print(f"\nTotal RAM: {memory.total / (1024**3):.2f} GB")
print(f"Available RAM: {memory.available / (1024**3):.2f} GB")

Test dataset memory usage:
145.91 MB

Total RAM: 15.73 GB
Available RAM: 1.95 GB


In [ ]:
import gc
import psutil

del test_df
gc.collect()

memory = psutil.virtual_memory()

print(f"Available RAM: {memory.available / (1024**3):.2f} GB")

Available RAM: 1.86 GB


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn

DATASET_DIR = Path("/content")

csv_files = sorted(DATASET_DIR.glob("*.csv"))

print("Dataset folder exists:", DATASET_DIR.exists())
print("CSV files found:", len(csv_files))

for file in csv_files:
    print(file.name)

Dataset folder exists: True
CSV files found: 8
Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Friday-WorkingHours-Morning.pcap_ISCX.csv
Monday-WorkingHours.pcap_ISCX.csv
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Tuesday-WorkingHours.pcap_ISCX.csv
Wednesday-workingHours.pcap_ISCX.csv


In [ ]:
import torch
from pathlib import Path

# Check GPU availability
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. Check Runtime > Change runtime type.")

# Check dataset files
DATASET_DIR = Path("/content")
csv_files = sorted(DATASET_DIR.glob("*.csv"))

print("\nCSV files found:", len(csv_files))
print(f"Total dataset size: {sum(f.stat().st_size for f in csv_files) / (1024**3):.2f} GB")

for file in csv_files:
    print(file.name)

GPU available: True
GPU name: Tesla T4

CSV files found: 8
Total dataset size: 0.48 GB
Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Friday-WorkingHours-Morning.pcap_ISCX.csv
Monday-WorkingHours.pcap_ISCX.csv
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Tuesday-WorkingHours.pcap_ISCX.csv
Wednesday-workingHours.pcap_ISCX.csv


In [ ]:
import pandas as pd

reference_columns = None
all_consistent = True

for file in csv_files:
    columns = pd.read_csv(file, nrows=0).columns.str.strip().tolist()

    print(f"{file.name}: {len(columns)} columns")

    if reference_columns is None:
        reference_columns = columns
    elif columns != reference_columns:
        all_consistent = False
        print("  WARNING: Column names or order differ!")

print("\nAll files consistent:", all_consistent)
print("Feature columns:", len(reference_columns) - 1)
print("Last column:", reference_columns[-1])

Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv: 79 columns
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv: 79 columns
Friday-WorkingHours-Morning.pcap_ISCX.csv: 79 columns
Monday-WorkingHours.pcap_ISCX.csv: 79 columns
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv: 79 columns
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv: 79 columns
Tuesday-WorkingHours.pcap_ISCX.csv: 79 columns
Wednesday-workingHours.pcap_ISCX.csv: 79 columns

All files consistent: True
Feature columns: 78
Last column: Label


In [ ]:
import numpy as np
import pandas as pd
from collections import Counter

def map_label(label):
    label = str(label).strip()

    if label == "BENIGN":
        return "Normal"
    if label.startswith("DoS "):
        return "DoS"
    if label == "DDoS":
        return "DDoS"
    if label == "PortScan":
        return "PortScan"
    if label in {"FTP-Patator", "SSH-Patator"}:
        return "BruteForce"
    if label.startswith("Web Attack"):
        return "WebAttack"
    if label == "Bot":
        return "Bot"
    if label == "Infiltration":
        return "Infiltration"
    if label == "Heartbleed":
        return "Heartbleed"

    return "Unknown"


CHUNK_SIZE = 100_000
cleaned_chunks = []
class_counts = Counter()
rows_read = 0
rows_removed = 0

for file in csv_files:
    print(f"\nProcessing: {file.name}")

    for chunk in pd.read_csv(
        file,
        chunksize=CHUNK_SIZE,
        low_memory=False,
    ):
        chunk.columns = chunk.columns.str.strip()

        label_col = next(
            col for col in chunk.columns
            if col.lower() == "label"
        )

        chunk["Target"] = chunk[label_col].map(map_label)
        chunk.drop(columns=[label_col], inplace=True)

        feature_cols = [
            col for col in chunk.columns if col != "Target"
        ]

        chunk[feature_cols] = chunk[feature_cols].apply(
            pd.to_numeric, errors="coerce"
        )

        chunk[feature_cols] = chunk[feature_cols].replace(
            [np.inf, -np.inf], np.nan
        )

        rows_read += len(chunk)

        valid_mask = (
            chunk["Target"].ne("Unknown")
            & chunk[feature_cols].notna().all(axis=1)
        )

        rows_removed += int((~valid_mask).sum())
        chunk = chunk.loc[valid_mask].copy()

        class_counts.update(chunk["Target"].value_counts().to_dict())
        cleaned_chunks.append(chunk)

    print(f"Rows read so far: {rows_read:,}")
    print(f"Valid rows retained: {sum(class_counts.values()):,}")

df = pd.concat(cleaned_chunks, ignore_index=True)

del cleaned_chunks

print("\nCleaning complete!")
print("Final shape:", df.shape)
print("Rows read:", f"{rows_read:,}")
print("Rows removed:", f"{rows_removed:,}")
print("Missing values:", int(df.isna().sum().sum()))
print("\nClass distribution:")
display(df["Target"].value_counts().to_frame("Count"))


Processing: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Rows read so far: 225,745
Valid rows retained: 225,711

Processing: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Rows read so far: 512,212
Valid rows retained: 511,807

Processing: Friday-WorkingHours-Morning.pcap_ISCX.csv
Rows read so far: 703,245
Valid rows retained: 702,718

Processing: Monday-WorkingHours.pcap_ISCX.csv
Rows read so far: 954,095
Valid rows retained: 953,353

Processing: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Rows read so far: 1,242,697
Valid rows retained: 1,241,748

Processing: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Rows read so far: 1,413,063
Valid rows retained: 1,411,979

Processing: Tuesday-WorkingHours.pcap_ISCX.csv
Rows read so far: 1,561,504
Valid rows retained: 1,560,318

Processing: Wednesday-workingHours.pcap_ISCX.csv
Rows read so far: 1,647,492
Valid rows retained: 1,646,209

Cleaning complete!
Final shape: (1646209, 79)
Rows read: 1,647,492
Rows re

,Count
Target,
Normal,1325135
PortScan,158804
DDoS,128025
DoS,22155
BruteForce,7918
WebAttack,2180
Bot,1956
Infiltration,36


In [ ]:
print("Original rows expected:", 2_830_743)
print("Rows read in Colab:", rows_read)
print("Difference:", 2_830_743 - rows_read)

print("\nAll target classes:")
print(sorted(df["Target"].unique()))

print("\nRare class counts:")
display(
    df["Target"]
    .value_counts()
    .reindex(
        [
            "Normal",
            "DoS",
            "DDoS",
            "PortScan",
            "BruteForce",
            "WebAttack",
            "Bot",
            "Infiltration",
            "Heartbleed",
        ],
        fill_value=0,
    )
    .rename("Count")
    .to_frame()
)

print("\nUploaded file sizes:")
for file in csv_files:
    print(f"{file.name}: {file.stat().st_size / (1024**2):.1f} MB")

Original rows expected: 2830743
Rows read in Colab: 1647492
Difference: 1183251

All target classes:
['Bot', 'BruteForce', 'DDoS', 'DoS', 'Infiltration', 'Normal', 'PortScan', 'WebAttack']

Rare class counts:


,Count
Target,
Normal,1325135
DoS,22155
DDoS,128025
PortScan,158804
BruteForce,7918
WebAttack,2180
Bot,1956
Infiltration,36
Heartbleed,0



Uploaded file sizes:
Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv: 73.6 MB
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv: 73.3 MB
Friday-WorkingHours-Morning.pcap_ISCX.csv: 55.6 MB
Monday-WorkingHours.pcap_ISCX.csv: 87.0 MB
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv: 79.3 MB
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv: 49.6 MB
Tuesday-WorkingHours.pcap_ISCX.csv: 47.0 MB
Wednesday-workingHours.pcap_ISCX.csv: 28.0 MB


In [ ]:
from collections import Counter
import pandas as pd

file_report = []

for file in csv_files:
    row_count = 0
    labels = Counter()

    print("Checking:", file.name)

    for chunk in pd.read_csv(file, chunksize=100_000, low_memory=False):
        chunk.columns = chunk.columns.str.strip()

        label_col = next(
            (c for c in chunk.columns if c.lower() == "label"),
            None,
        )

        if label_col is None:
            raise ValueError(f"Label column missing in {file.name}")

        row_count += len(chunk)
        labels.update(chunk[label_col].astype("string").str.strip().dropna())

    file_report.append({
        "File": file.name,
        "Rows": row_count,
        "Size (MB)": round(file.stat().st_size / (1024**2), 1),
        "Heartbleed rows": labels.get("Heartbleed", 0),
        "Infiltration rows": labels.get("Infiltration", 0),
    })

display(pd.DataFrame(file_report))
print("Total rows across files:", sum(x["Rows"] for x in file_report))

Checking: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Checking: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Checking: Friday-WorkingHours-Morning.pcap_ISCX.csv
Checking: Monday-WorkingHours.pcap_ISCX.csv
Checking: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Checking: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Checking: Tuesday-WorkingHours.pcap_ISCX.csv
Checking: Wednesday-workingHours.pcap_ISCX.csv


,File,Rows,Size (MB),Heartbleed rows,Infiltration rows
0,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,225745,73.6,0,0
1,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,286467,73.3,0,0
2,Friday-WorkingHours-Morning.pcap_ISCX.csv,191033,55.6,0,0
3,Monday-WorkingHours.pcap_ISCX.csv,250850,87.0,0,0
4,Thursday-WorkingHours-Afternoon-Infilteration....,288602,79.3,0,36
5,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,170366,49.6,0,0
6,Tuesday-WorkingHours.pcap_ISCX.csv,148441,47.0,0,0
7,Wednesday-workingHours.pcap_ISCX.csv,85988,28.0,0,0


Total rows across files: 1647492


In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive")
DATASET_DIR = DRIVE_ROOT / "CIC-IDS2017"

print("Folder exists:", DATASET_DIR.exists())

if DATASET_DIR.exists():
    print("Contents:")
    for item in DATASET_DIR.iterdir():
        print(item.name, "—", "Folder" if item.is_dir() else "File")

Mounted at /content/drive
Folder exists: True
Contents:
MachineLearningCVE — Folder


In [ ]:
from pathlib import Path

DATASET_DIR = (
    Path("/content/drive/MyDrive/CIC-IDS2017/MachineLearningCVE")
)

csv_files = sorted(DATASET_DIR.glob("*.csv"))

print("Dataset folder exists:", DATASET_DIR.exists())
print("CSV files found:", len(csv_files))

for file in csv_files:
    print(f"{file.name}: {file.stat().st_size / (1024**2):.1f} MB")

Dataset folder exists: True
CSV files found: 8
Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv: 73.6 MB
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv: 73.3 MB
Friday-WorkingHours-Morning.pcap_ISCX.csv: 55.6 MB
Monday-WorkingHours.pcap_ISCX.csv: 168.7 MB
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv: 79.3 MB
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv: 49.6 MB
Tuesday-WorkingHours.pcap_ISCX.csv: 128.8 MB
Wednesday-workingHours.pcap_ISCX.csv: 214.7 MB


In [ ]:
from collections import Counter
import pandas as pd

total_rows = 0
label_counts = Counter()

for file in csv_files:
    file_rows = 0
    print("Checking:", file.name)

    for chunk in pd.read_csv(file, chunksize=100_000, low_memory=False):
        chunk.columns = chunk.columns.str.strip()

        label_col = next(
            col for col in chunk.columns
            if col.lower() == "label"
        )

        file_rows += len(chunk)
        label_counts.update(
            chunk[label_col].astype("string").str.strip().dropna()
        )

    total_rows += file_rows
    print(f"  Rows: {file_rows:,}")

print(f"\nTotal rows: {total_rows:,}")
print(f"Unique original labels: {len(label_counts)}")

print("\nOriginal label counts:")
display(
    pd.DataFrame(
        label_counts.items(),
        columns=["Original Label", "Count"],
    ).sort_values("Count", ascending=False).reset_index(drop=True)
)

print("\nHeartbleed rows:", label_counts.get("Heartbleed", 0))

Checking: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
  Rows: 225,745
Checking: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
  Rows: 286,467
Checking: Friday-WorkingHours-Morning.pcap_ISCX.csv
  Rows: 191,033
Checking: Monday-WorkingHours.pcap_ISCX.csv
  Rows: 529,918
Checking: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
  Rows: 288,602
Checking: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
  Rows: 170,366
Checking: Tuesday-WorkingHours.pcap_ISCX.csv
  Rows: 445,909
Checking: Wednesday-workingHours.pcap_ISCX.csv
  Rows: 692,703

Total rows: 2,830,743
Unique original labels: 15

Original label counts:


,Original Label,Count
0,BENIGN,2273097
1,DoS Hulk,231073
2,PortScan,158930
3,DDoS,128027
4,DoS GoldenEye,10293
5,FTP-Patator,7938
6,SSH-Patator,5897
7,DoS slowloris,5796
8,DoS Slowhttptest,5499
9,Bot,1966



Heartbleed rows: 11


In [ ]:
import gc
import numpy as np
import pandas as pd
from collections import Counter

def map_label(label):
    label = str(label).strip()

    if label == "BENIGN":
        return "Normal"
    if label.startswith("DoS "):
        return "DoS"
    if label == "DDoS":
        return "DDoS"
    if label == "PortScan":
        return "PortScan"
    if label in {"FTP-Patator", "SSH-Patator"}:
        return "BruteForce"
    if label.startswith("Web Attack"):
        return "WebAttack"
    if label == "Bot":
        return "Bot"
    if label == "Infiltration":
        return "Infiltration"
    if label == "Heartbleed":
        return "Heartbleed"

    return "Unknown"


CHUNK_SIZE = 100_000
cleaned_chunks = []
rows_read = 0
rows_removed = 0

for file in csv_files:
    print(f"\nProcessing: {file.name}")

    for chunk in pd.read_csv(
        file,
        chunksize=CHUNK_SIZE,
        low_memory=False,
    ):
        chunk.columns = chunk.columns.str.strip()

        label_col = next(
            col for col in chunk.columns
            if col.lower() == "label"
        )

        chunk["Target"] = chunk[label_col].map(map_label)
        chunk.drop(columns=[label_col], inplace=True)

        feature_cols = [
            col for col in chunk.columns if col != "Target"
        ]

        chunk[feature_cols] = chunk[feature_cols].apply(
            pd.to_numeric, errors="coerce"
        )

        chunk[feature_cols] = chunk[feature_cols].replace(
            [np.inf, -np.inf], np.nan
        )

        rows_read += len(chunk)

        valid_mask = (
            chunk["Target"].ne("Unknown")
            & chunk[feature_cols].notna().all(axis=1)
        )

        rows_removed += int((~valid_mask).sum())
        cleaned_chunks.append(chunk.loc[valid_mask].copy())

    print(f"Rows read so far: {rows_read:,}")

df = pd.concat(cleaned_chunks, ignore_index=True)
del cleaned_chunks
gc.collect()

print("\nCleaning complete!")
print("Rows read:", f"{rows_read:,}")
print("Rows removed:", f"{rows_removed:,}")
print("Cleaned dataset shape:", df.shape)
print("\nClass distribution:")
display(df["Target"].value_counts().rename("Count").to_frame())


Processing: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Rows read so far: 225,745

Processing: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Rows read so far: 512,212

Processing: Friday-WorkingHours-Morning.pcap_ISCX.csv
Rows read so far: 703,245

Processing: Monday-WorkingHours.pcap_ISCX.csv
Rows read so far: 1,233,163

Processing: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Rows read so far: 1,521,765

Processing: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Rows read so far: 1,692,131

Processing: Tuesday-WorkingHours.pcap_ISCX.csv
Rows read so far: 2,138,040

Processing: Wednesday-workingHours.pcap_ISCX.csv
Rows read so far: 2,830,743

Cleaning complete!
Rows read: 2,830,743
Rows removed: 2,867
Cleaned dataset shape: (2827876, 79)

Class distribution:


,Count
Target,
Normal,2271320
DoS,251712
PortScan,158804
DDoS,128025
BruteForce,13832
WebAttack,2180
Bot,1956
Infiltration,36
Heartbleed,11


In [ ]:
import psutil
import torch
import gc

gc.collect()

print(f"Dataset memory: {df.memory_usage(deep=True).sum() / (1024**3):.2f} GB")

ram = psutil.virtual_memory()
print(f"Available system RAM: {ram.available / (1024**3):.2f} GB")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU total memory: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
    print(f"GPU allocated memory: {torch.cuda.memory_allocated(0) / (1024**3):.2f} GB")

Dataset memory: 1.79 GB
Available system RAM: 7.81 GB
GPU: Tesla T4
GPU total memory: 14.56 GB
GPU allocated memory: 0.00 GB


In [ ]:
%pip install -q xgboost

import xgboost as xgb
import torch

print("XGBoost version:", xgb.__version__)
print("GPU available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

XGBoost version: 3.4.1
GPU available: True
GPU: Tesla T4


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Separate features and target
X = df.drop(columns=["Target"])
y_text = df["Target"]

# Encode the 9 target categories as integers
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_text)

print("Class-to-number mapping:")
for number, name in enumerate(label_encoder.classes_):
    print(f"{number}: {name}")

# Split without discarding rows
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print("\nTraining rows:", f"{len(X_train):,}")
print("Testing rows:", f"{len(X_test):,}")
print("Total rows:", f"{len(X_train) + len(X_test):,}")

print("\nTraining class counts:")
display(
    pd.Series(
        label_encoder.inverse_transform(y_train)
    ).value_counts().rename("Count").to_frame()
)

print("\nTesting class counts:")
display(
    pd.Series(
        label_encoder.inverse_transform(y_test)
    ).value_counts().rename("Count").to_frame()
)

Class-to-number mapping:
0: Bot
1: BruteForce
2: DDoS
3: DoS
4: Heartbleed
5: Infiltration
6: Normal
7: PortScan
8: WebAttack

Training rows: 2,262,300
Testing rows: 565,576
Total rows: 2,827,876

Training class counts:


,Count
Normal,1817055
DoS,201369
PortScan,127043
DDoS,102420
BruteForce,11066
WebAttack,1744
Bot,1565
Infiltration,29
Heartbleed,9



Testing class counts:


,Count
Normal,454265
DoS,50343
PortScan,31761
DDoS,25605
BruteForce,2766
WebAttack,436
Bot,391
Infiltration,7
Heartbleed,2


In [ ]:
import numpy as np
import gc

# Convert features to compact numeric arrays
X_train_np = X_train.to_numpy(dtype=np.float32, copy=True)
X_test_np = X_test.to_numpy(dtype=np.float32, copy=True)

y_train_np = y_train.astype(np.int32, copy=False)
y_test_np = y_test.astype(np.int32, copy=False)

print("Training features:", X_train_np.shape, X_train_np.dtype)
print("Testing features:", X_test_np.shape, X_test_np.dtype)

print(
    "Training array size:",
    f"{X_train_np.nbytes / (1024**3):.2f} GB"
)
print(
    "Testing array size:",
    f"{X_test_np.nbytes / (1024**3):.2f} GB"
)

print("Non-finite training values:", int(
    (~np.isfinite(X_train_np)).sum()
))
print("Non-finite testing values:", int(
    (~np.isfinite(X_test_np)).sum()
))

Training features: (2262300, 78) float32
Testing features: (565576, 78) float32
Training array size: 0.66 GB
Testing array size: 0.16 GB
Non-finite training values: 0
Non-finite testing values: 0


In [ ]:
import xgboost as xgb

NUM_CLASSES = len(label_encoder.classes_)

model = xgb.XGBClassifier(
    objective="multi:softprob",
    num_class=NUM_CLASSES,
    eval_metric="mlogloss",
    tree_method="hist",
    device="cuda",
    n_estimators=300,
    learning_rate=0.1,
    max_depth=8,
    min_child_weight=2,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=4,
    early_stopping_rounds=20,
)

print("Model configured.")
print("Number of classes:", NUM_CLASSES)
print("Training device:", "cuda")
print("Maximum boosting rounds:", model.get_params()["n_estimators"])

Model configured.
Number of classes: 9
Training device: cuda
Maximum boosting rounds: 300


In [ ]:
import time

start_time = time.time()

model.fit(
    X_train_np,
    y_train_np,
    eval_set=[(X_test_np, y_test_np)],
    verbose=25,
)

elapsed = time.time() - start_time

print(f"\nTraining finished in {elapsed / 60:.2f} minutes.")
print("Best boosting round:", model.best_iteration)
print("Best validation log loss:", model.best_score)

[0]	validation_0-mlogloss:0.52251
[25]	validation_0-mlogloss:0.03428
[50]	validation_0-mlogloss:0.00656
[75]	validation_0-mlogloss:0.00372
[100]	validation_0-mlogloss:0.00319
[125]	validation_0-mlogloss:0.00299
[150]	validation_0-mlogloss:0.00290
[175]	validation_0-mlogloss:0.00285
[200]	validation_0-mlogloss:0.00283
[225]	validation_0-mlogloss:0.00283
[232]	validation_0-mlogloss:0.00284

Training finished in 1.62 minutes.
Best boosting round: 212
Best validation log loss: 0.00282669982380631


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)

# Predict using the best iteration
y_pred = model.predict(X_test_np)

print(f"Test accuracy: {accuracy_score(y_test_np, y_pred):.6f}")

print("\nClassification report:")
print(
    classification_report(
        y_test_np,
        y_pred,
        labels=list(range(len(label_encoder.classes_))),
        target_names=label_encoder.classes_,
        zero_division=0,
        digits=4,
    )
)

print("\nConfusion matrix:")
print(confusion_matrix(y_test_np, y_pred))

/usr/local/lib/python3.13/dist-packages/xgboost/core.py:569: UserWarning: [14:21:23] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


Test accuracy: 0.999254

Classification report:
              precision    recall  f1-score   support

         Bot     0.9065    0.7442    0.8174       391
  BruteForce     0.9993    0.9993    0.9993      2766
        DDoS     0.9999    0.9999    0.9999     25605
         DoS     0.9986    0.9996    0.9991     50343
  Heartbleed     1.0000    0.5000    0.6667         2
Infiltration     1.0000    0.7143    0.8333         7
      Normal     0.9997    0.9994    0.9995    454265
    PortScan     0.9944    0.9996    0.9970     31761
   WebAttack     0.9908    0.9839    0.9873       436

    accuracy                         0.9993    565576
   macro avg     0.9877    0.8823    0.9222    565576
weighted avg     0.9992    0.9993    0.9992    565576


Confusion matrix:
[[   291      0      0      0      0      0    100      0      0]
 [     0   2764      0      0      0      0      2      0      0]
 [     0      0  25602      0      0      0      3      0      0]
 [     0      0      0  50323 

In [ ]:
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report

dtest = xgb.DMatrix(X_test_np, label=y_test_np)
y_pred = model.get_booster().predict(dtest).argmax(axis=1)

print(f"Test accuracy: {accuracy_score(y_test_np, y_pred):.6%}")

print(classification_report(
    y_test_np,
    y_pred,
    labels=list(range(len(label_encoder.classes_))),
    target_names=label_encoder.classes_,
    zero_division=0,
    digits=4,
))

Test accuracy: 99.925563%
              precision    recall  f1-score   support

         Bot     0.8985    0.7468    0.8156       391
  BruteForce     0.9993    0.9993    0.9993      2766
        DDoS     0.9999    0.9999    0.9999     25605
         DoS     0.9986    0.9996    0.9991     50343
  Heartbleed     1.0000    0.5000    0.6667         2
Infiltration     1.0000    0.7143    0.8333         7
      Normal     0.9997    0.9994    0.9995    454265
    PortScan     0.9944    0.9997    0.9970     31761
   WebAttack     0.9908    0.9862    0.9885       436

    accuracy                         0.9993    565576
   macro avg     0.9868    0.8828    0.9221    565576
weighted avg     0.9992    0.9993    0.9992    565576



In [ ]:
from sklearn.metrics import accuracy_score, f1_score

# Predict on the training set
y_train_pred = model.predict(X_train_np)

# Compare training and test performance
train_accuracy = accuracy_score(y_train_np, y_train_pred)
test_accuracy = accuracy_score(y_test_np, y_pred)

train_f1 = f1_score(
    y_train_np, y_train_pred, average="macro"
)
test_f1 = f1_score(
    y_test_np, y_pred, average="macro"
)

print(f"Training accuracy: {train_accuracy:.6%}")
print(f"Testing accuracy:  {test_accuracy:.6%}")
print(f"Accuracy gap:      {(train_accuracy - test_accuracy) * 100:.4f} percentage points")

print(f"\nTraining macro F1: {train_f1:.6f}")
print(f"Testing macro F1:  {test_f1:.6f}")
print(f"Macro F1 gap:      {train_f1 - test_f1:.6f}")

Training accuracy: 99.939177%
Testing accuracy:  99.925563%
Accuracy gap:      0.0136 percentage points

Training macro F1: 0.984177
Testing macro F1:  0.922113
Macro F1 gap:      0.062064


In [ ]:
from collections import Counter
import pandas as pd

label_by_file = []

for file in csv_files:
    counts = Counter()

    for chunk in pd.read_csv(file, chunksize=100_000, low_memory=False):
        chunk.columns = chunk.columns.str.strip()
        label_col = next(c for c in chunk.columns if c.lower() == "label")
        mapped = chunk[label_col].map(map_label)
        counts.update(mapped[mapped != "Unknown"].dropna())

    for label, count in counts.items():
        label_by_file.append({
            "File": file.name,
            "Class": label,
            "Count": count,
        })

file_class_counts = pd.DataFrame(label_by_file)

display(
    file_class_counts.pivot_table(
        index="File",
        columns="Class",
        values="Count",
        aggfunc="sum",
        fill_value=0,
    )
)

Class,Bot,BruteForce,DDoS,DoS,Heartbleed,Infiltration,Normal,PortScan,WebAttack
File,,,,,,,,,
Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,0,0,128027,0,0,0,97718,0,0
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv,0,0,0,0,0,0,127537,158930,0
Friday-WorkingHours-Morning.pcap_ISCX.csv,1966,0,0,0,0,0,189067,0,0
Monday-WorkingHours.pcap_ISCX.csv,0,0,0,0,0,0,529918,0,0
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv,0,0,0,0,0,36,288566,0,0
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv,0,0,0,0,0,0,168186,0,2180
Tuesday-WorkingHours.pcap_ISCX.csv,0,13835,0,0,0,0,432074,0,0
Wednesday-workingHours.pcap_ISCX.csv,0,0,0,252661,11,0,440031,0,0


In [ ]:
import pandas as pd

importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model.feature_importances_,
}).sort_values("Importance", ascending=False)

print("Top 20 most important features:")
display(importance.head(20).reset_index(drop=True))

Top 20 most important features:


,Feature,Importance
0,Bwd Packet Length Mean,0.098103
1,Bwd Packet Length Std,0.093704
2,Average Packet Size,0.082724
3,PSH Flag Count,0.082708
4,act_data_pkt_fwd,0.079021
5,Packet Length Variance,0.058980
6,Idle Mean,0.055208
7,Idle Max,0.051411
8,Subflow Bwd Bytes,0.050564
9,Subflow Fwd Bytes,0.035581


In [ ]:
import xgboost as xgb
from sklearn.metrics import accuracy_score, f1_score

top5 = importance.head(5)["Feature"].tolist()
print("Removing these features:", top5)

X_train_reduced = X_train.drop(columns=top5).to_numpy(dtype="float32")
X_test_reduced = X_test.drop(columns=top5).to_numpy(dtype="float32")

ablation_model = xgb.XGBClassifier(
    objective="multi:softprob",
    num_class=len(label_encoder.classes_),
    eval_metric="mlogloss",
    tree_method="hist",
    device="cuda",
    n_estimators=300,
    learning_rate=0.1,
    max_depth=8,
    min_child_weight=2,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=4,
    early_stopping_rounds=20,
)

ablation_model.fit(
    X_train_reduced,
    y_train_np,
    eval_set=[(X_test_reduced, y_test_np)],
    verbose=False,
)

reduced_pred = ablation_model.predict(X_test_reduced)

print(f"\nOriginal test accuracy: {accuracy_score(y_test_np, y_pred):.6%}")
print(f"Reduced test accuracy:  {accuracy_score(y_test_np, reduced_pred):.6%}")
print(f"Original macro F1: {f1_score(y_test_np, y_pred, average='macro'):.6f}")
print(f"Reduced macro F1:  {f1_score(y_test_np, reduced_pred, average='macro'):.6f}")

Removing these features: ['Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Average Packet Size', 'PSH Flag Count', 'act_data_pkt_fwd']

Original test accuracy: 99.925563%
Reduced test accuracy:  99.926093%
Original macro F1: 0.922113
Reduced macro F1:  0.923218


In [ ]:
from sklearn.metrics import classification_report

print(classification_report(
    y_test_np,
    reduced_pred,
    labels=list(range(len(label_encoder.classes_))),
    target_names=label_encoder.classes_,
    zero_division=0,
    digits=4,
))

              precision    recall  f1-score   support

         Bot     0.9136    0.7570    0.8280       391
  BruteForce     0.9996    0.9989    0.9993      2766
        DDoS     0.9999    0.9999    0.9999     25605
         DoS     0.9985    0.9996    0.9990     50343
  Heartbleed     1.0000    0.5000    0.6667         2
Infiltration     1.0000    0.7143    0.8333         7
      Normal     0.9997    0.9994    0.9996    454265
    PortScan     0.9945    0.9996    0.9970     31761
   WebAttack     0.9907    0.9817    0.9862       436

    accuracy                         0.9993    565576
   macro avg     0.9885    0.8834    0.9232    565576
weighted avg     0.9993    0.9993    0.9993    565576



In [ ]:
# Use the class counts already calculated in file_class_counts

class_file_presence = (
    file_class_counts
    .pivot_table(
        index="Class",
        columns="File",
        values="Count",
        aggfunc="sum",
        fill_value=0,
    )
    .gt(0)
)

summary = pd.DataFrame({
    "Files containing class": class_file_presence.sum(axis=1),
    "Can hold out a file while keeping class in training":
        class_file_presence.sum(axis=1) >= 2,
})

display(summary)

,Files containing class,Can hold out a file while keeping class in training
Class,,
Bot,1,False
BruteForce,1,False
DDoS,1,False
DoS,1,False
Heartbleed,1,False
Infiltration,1,False
Normal,8,True
PortScan,1,False
WebAttack,1,False


In [ ]:
from sklearn.metrics import accuracy_score, f1_score

reduced_train_pred = ablation_model.predict(X_train_reduced)

print(
    "Reduced model training accuracy:",
    f"{accuracy_score(y_train_np, reduced_train_pred):.6%}"
)
print(
    "Reduced model test accuracy:",
    f"{accuracy_score(y_test_np, reduced_pred):.6%}"
)

print(
    "Reduced model training macro F1:",
    f"{f1_score(y_train_np, reduced_train_pred, average='macro'):.6f}"
)
print(
    "Reduced model test macro F1:",
    f"{f1_score(y_test_np, reduced_pred, average='macro'):.6f}"
)

Reduced model training accuracy: 99.937807%
Reduced model test accuracy: 99.926093%
Reduced model training macro F1: 0.984016
Reduced model test macro F1: 0.923218


In [ ]:
from pathlib import Path
import json
import joblib

ARTIFACT_DIR = Path(
    "/content/drive/MyDrive/NIDS/artifacts"
)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# Save the reduced-feature model
model_path = ARTIFACT_DIR / "nids_xgboost_reduced.json"
ablation_model.save_model(str(model_path))

# Save the exact feature order used by the reduced model
reduced_features = X_train.drop(columns=top5).columns.tolist()

metadata = {
    "model_type": "XGBClassifier",
    "num_classes": len(label_encoder.classes_),
    "class_names": label_encoder.classes_.tolist(),
    "features": reduced_features,
    "removed_features": top5,
    "dataset": "CIC-IDS2017 MachineLearningCVE",
    "training_rows": int(len(X_train_np)),
    "testing_rows": int(len(X_test_np)),
}

with open(ARTIFACT_DIR / "nids_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

# Save label encoder separately
joblib.dump(
    label_encoder,
    ARTIFACT_DIR / "label_encoder.joblib",
)

print("Artifacts saved to:", ARTIFACT_DIR)
for path in ARTIFACT_DIR.iterdir():
    print(path.name, f"({path.stat().st_size / 1024:.1f} KB)")

Artifacts saved to: /content/drive/MyDrive/NIDS/artifacts
nids_xgboost_reduced.json (5705.9 KB)
nids_metadata.json (2.2 KB)
label_encoder.joblib (0.6 KB)


In [ ]:

from pathlib import Path
import json
import joblib
import xgboost as xgb
import numpy as np

ARTIFACT_DIR = Path("/content/drive/MyDrive/NIDS/artifacts")

# 1. Load model
loaded_model = xgb.XGBClassifier()
loaded_model.load_model(
    str(ARTIFACT_DIR / "nids_xgboost_reduced.json")
)

# 2. Load metadata and label encoder
with open(ARTIFACT_DIR / "nids_metadata.json", "r") as f:
    metadata = json.load(f)

loaded_encoder = joblib.load(
    ARTIFACT_DIR / "label_encoder.joblib"
)

# 3. Verify feature and class information
print("Model loaded successfully")
print("Model expects features:", loaded_model.get_booster().num_features())
print("Metadata feature count:", len(metadata["features"]))
print("Number of classes:", len(metadata["class_names"]))
print("Class names:", metadata["class_names"])
print("Encoder classes:", loaded_encoder.classes_.tolist())

# 4. Test a prediction using one reduced-feature test row
X_sample = X_test.drop(
    columns=metadata["removed_features"]
).iloc[:1].to_numpy(dtype=np.float32)

prediction = loaded_model.predict(X_sample)
probabilities = loaded_model.predict_proba(X_sample)

print("\nPrediction test passed")
print("Predicted class:", loaded_encoder.inverse_transform(prediction)[0])
print("Confidence:", round(float(probabilities.max()) * 100, 2), "%")
print("Probability output shape:", probabilities.shape)


Model loaded successfully
Model expects features: 73
Metadata feature count: 73
Number of classes: 9
Class names: ['Bot', 'BruteForce', 'DDoS', 'DoS', 'Heartbleed', 'Infiltration', 'Normal', 'PortScan', 'WebAttack']
Encoder classes: ['Bot', 'BruteForce', 'DDoS', 'DoS', 'Heartbleed', 'Infiltration', 'Normal', 'PortScan', 'WebAttack']

Prediction test passed
Predicted class: Normal
Confidence: 100.0 %
Probability output shape: (1, 9)


In [ ]:

from google.colab import files
from pathlib import Path
import shutil

ARTIFACT_DIR = Path("/content/drive/MyDrive/NIDS/artifacts")
ZIP_PATH = Path("/content/nids_artifacts.zip")

shutil.make_archive(
    str(ZIP_PATH).removesuffix(".zip"),
    "zip",
    ARTIFACT_DIR
)

files.download(str(ZIP_PATH))


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
print("DataFrame shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

DataFrame shape: (2827876, 79)

Column names:
['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG 

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Target
0,54865,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,Normal
1,55054,109,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,Normal
2,55055,52,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,Normal
3,46236,34,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,Normal
4,54863,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,Normal


In [ ]:

import numpy as np
import pandas as pd
import json
import xgboost as xgb
import joblib
from pathlib import Path

ARTIFACT_DIR = Path("/content/drive/MyDrive/NIDS/artifacts")

# Load saved model and metadata
model = xgb.XGBClassifier()
model.load_model(str(ARTIFACT_DIR / "nids_xgboost_reduced.json"))

with open(ARTIFACT_DIR / "nids_metadata.json", "r") as f:
    metadata = json.load(f)

label_encoder = joblib.load(ARTIFACT_DIR / "label_encoder.joblib")
features = metadata["features"]

# Select a real Normal sample
sample = df.loc[df["Target"] == "Normal"].iloc[0]

# Keep the exact feature order used during training
X_sample = sample[features].to_frame().T.astype(np.float32)

true_label = sample["Target"]
predicted_index = int(model.predict(X_sample)[0])
probabilities = model.predict_proba(X_sample)[0]
predicted_label = label_encoder.inverse_transform(
    [predicted_index]
)[0]

print("True label:", true_label)
print("Predicted label:", predicted_label)
print(f"Confidence: {probabilities.max() * 100:.4f}%")
print("Correct prediction:", true_label == predicted_label)
print("Feature count:", X_sample.shape[1])


True label: Normal
Predicted label: Normal
Confidence: 99.9999%
Correct prediction: True
Feature count: 73


In [ ]:

print("Selected feature count:", len(features))
print("Sample shape:", X_sample.shape)

assert len(features) == 73
assert X_sample.shape == (1, 73)

# Export a real sample for the local FastAPI endpoint
request_data = {
    "flow_id": "real-normal-test-001",
    **{
        name: float(X_sample.iloc[0][name])
        for name in features
    },
}

output_path = "/content/real_normal_test.json"

with open(output_path, "w") as f:
    json.dump(request_data, f, indent=2)

print("Exported:", output_path)
print("Feature count:", len(request_data) - 1)


Selected feature count: 73
Sample shape: (1, 73)
Exported: /content/real_normal_test.json
Feature count: 73


In [ ]:
from google.colab import files
files.download("/content/real_normal_test.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:

# Select a real PortScan sample
attack_sample = df.loc[df["Target"] == "PortScan"].iloc[0]

X_attack = attack_sample[features].to_frame().T.astype(np.float32)
true_label = attack_sample["Target"]

predicted_index = int(model.predict(X_attack)[0])
probabilities = model.predict_proba(X_attack)[0]
predicted_label = label_encoder.inverse_transform(
    [predicted_index]
)[0]

print("True label:", true_label)
print("Predicted label:", predicted_label)
print(f"Confidence: {probabilities.max() * 100:.4f}%")
print("Correct prediction:", true_label == predicted_label)
print("Feature count:", X_attack.shape[1])


True label: PortScan
Predicted label: PortScan
Confidence: 85.0309%
Correct prediction: True
Feature count: 73


In [ ]:

request_data = {
    "flow_id": "real-portscan-test-001",
    **{
        name: float(X_attack.iloc[0][name])
        for name in features
    },
}

output_path = "/content/real_portscan_test.json"

with open(output_path, "w") as f:
    json.dump(request_data, f, indent=2)

print("Exported:", output_path)
print("Feature count:", len(request_data) - 1)


Exported: /content/real_portscan_test.json
Feature count: 73


In [ ]:

request_data = {
    "flow_id": "real-portscan-test-001",
    **{
        name: float(X_attack.iloc[0][name])
        for name in features
    },
}

output_path = "/content/real_portscan_test.json"

with open(output_path, "w") as f:
    json.dump(request_data, f, indent=2)

print("Exported:", output_path)
print("Feature count:", len(request_data) - 1)


Exported: /content/real_portscan_test.json
Feature count: 73


In [ ]:
from google.colab import files
files.download("/content/real_portscan_test.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>